# RCDM Training on Google Colab

This notebook trains the RCDM (Representation-Conditioned Diffusion Model) on Tiny ImageNet using Google Colab's free GPU.

## Setup Steps:
1. **Runtime → Change runtime type → GPU (T4 recommended)**
2. Run cells top to bottom
3. When prompted, sign in to Google Drive (for saving checkpoints)
4. Upload `train_reps.pt` to Google Drive beforehand for fastest data access

## Repo:
[github.com/SeverinLe/master_implementation](https://github.com/SeverinLe/master_implementation)

## 1. Check GPU Availability

In [12]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB moin")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Memory: 101.97 GB moin


## 2. Mount Google Drive (Optional - for storing checkpoints)

In [13]:
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory in Drive
!mkdir -p /content/drive/MyDrive/rcdm_checkpoints

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Clone Code from GitHub

In [14]:
import os
import shutil

REPO_URL = "https://github.com/SeverinLe/master_implementation.git"
REPO_DIR = "/content/master_implementation"
OLD_DIR  = "/master_implementation"   # path used by earlier sessions

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo found at correct location, pulling latest changes...")
    !git -C {REPO_DIR} pull

elif os.path.isdir(os.path.join(OLD_DIR, ".git")):
    # Repo was cloned to the old root-level path — migrate it
    print(f"Migrating repo from {OLD_DIR} → {REPO_DIR} ...")
    # Preserve data/ subfolder if it already exists in the destination
    data_backup = None
    if os.path.isdir(os.path.join(REPO_DIR, "data")):
        data_backup = "/tmp/_rcdm_data_backup"
        shutil.copytree(os.path.join(REPO_DIR, "data"), data_backup, dirs_exist_ok=True)
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    shutil.copytree(OLD_DIR, REPO_DIR)
    if data_backup:
        shutil.copytree(data_backup, os.path.join(REPO_DIR, "data"), dirs_exist_ok=True)
    !git -C {REPO_DIR} pull

else:
    print(f"Cloning {REPO_URL} ...")
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

print("\nProject structure:")
for item in sorted(os.listdir(REPO_DIR)):
    print(f"  {item}")

Migrating repo from /master_implementation → /content/master_implementation ...
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 12 (delta 7), reused 9 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 5.20 KiB | 5.20 MiB/s, done.
From https://github.com/SeverinLe/master_implementation
   7169672..1f593b2  main       -> origin/main
Updating 7169672..1f593b2
Fast-forward
 colab_training.ipynb            | 356 +++++++++++++---------------------------
 guided_diffusion/pyproject.toml |   3 +
 guided_diffusion/setup.py       |   4 +-
 3 files changed, 120 insertions(+), 243 deletions(-)
 create mode 100644 guided_diffusion/pyproject.toml
/content/master_implementation

Project structure:
  .git
  .gitignore
  COLAB_SETUP_GUIDE.md
  colab_training.ipynb
  data
  gu

> **Tip:** If you push new changes from Cursor, re-run the cell above to `git pull` them into Colab without restarting the runtime.

In [15]:
# Verify the clone contains all expected modules
!ls rcdm/ scripts/ guided_diffusion/guided_diffusion/

guided_diffusion/guided_diffusion/:
dist_util.py	       __init__.py  __pycache__     train_util.py
fp16_util.py	       logger.py    resample.py     unet.py
gaussian_diffusion.py  losses.py    respace.py
image_datasets.py      nn.py	    script_util.py

rcdm/:
conditioning.py  dataset.py  encoder.py  __init__.py  test_conditioning.py

scripts/:
pipeline.py  precompute_reps.py  sampling.py  train.py


## 4. Install Dependencies

In [16]:
# Install guided_diffusion in editable mode (absolute path avoids cwd issues)
!pip install -e /content/master_implementation/guided_diffusion/ -q

# Other dependencies
!pip install "blobfile>=1.0.5" tqdm pillow torchvision -q

print("✓ Dependencies installed")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for guided-diffusion (pyproject.toml) ... done
✓ Dependencies installed


## 5. Get Training Data

`train_reps.pt` (~826MB) is not stored in GitHub (too large). Two options:

- **Option A (recommended):** Upload it to Google Drive once, then copy with the cell below — fast and persists across sessions.
- **Option B:** Upload directly to Colab — works but takes longer and is lost when the session ends.

In [17]:
# Mount Drive and copy train_reps.pt to the expected location

import os
import shutil
import torch
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = "/content/master_implementation"
DATA_DIR = os.path.join(REPO_DIR, "data", "tiny-imagenet-200")
os.makedirs(DATA_DIR, exist_ok=True)

SRC_PATH = "/content/drive/MyDrive/master_implementation/train_reps.pt"
DST_PATH = os.path.join(DATA_DIR, "train_reps.pt")

assert os.path.exists(SRC_PATH), f"Source file not found: {SRC_PATH}"

shutil.copy2(SRC_PATH, DST_PATH)

obj = torch.load(DST_PATH, map_location="cpu")
print("Loaded OK:", DST_PATH)
print("Type:", type(obj))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded OK: /content/master_implementation/data/tiny-imagenet-200/train_reps.pt
Type: <class 'dict'>


In [ ]:
# Download Tiny ImageNet images (~236 MB)
# train_reps.pt stores relative paths like data/tiny-imagenet-200/train/.../image.JPEG
# so the images must be present at that exact location relative to the repo root.

import os

REPO_DIR  = "/content/master_implementation"
TRAIN_DIR = os.path.join(REPO_DIR, "data", "tiny-imagenet-200", "train")

if os.path.isdir(TRAIN_DIR) and len(os.listdir(TRAIN_DIR)) >= 200:
    print(f"Tiny ImageNet already present ({len(os.listdir(TRAIN_DIR))} class folders)")
else:
    print("Downloading Tiny ImageNet (~236 MB)...")
    !wget -q --show-progress "http://cs231n.stanford.edu/tiny-imagenet-200.zip" \
        -O /tmp/tiny-imagenet.zip
    print("Extracting...")
    !unzip -q /tmp/tiny-imagenet.zip -d {REPO_DIR}/data/
    !rm /tmp/tiny-imagenet.zip
    print(f"Done. Class folders: {len(os.listdir(TRAIN_DIR))}")

In [18]:
# To save train_reps.pt to your Drive for future sessions (run once after uploading):
# !cp data/tiny-imagenet-200/train_reps.pt /content/drive/MyDrive/train_reps.pt

## 6. Verify Setup

In [19]:
import os
import sys

REPO_DIR = "/content/master_implementation"

# repo root → needed for rcdm
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# guided_diffusion parent → needed so unet.py can reach rcdm (both live under REPO_DIR)
GD_PARENT = os.path.join(REPO_DIR, "guided_diffusion")
if GD_PARENT not in sys.path:
    sys.path.insert(0, GD_PARENT)

# Test imports
from guided_diffusion.script_util import create_model_and_diffusion
from rcdm.dataset import RepresentationDataset
from rcdm.encoder import load_encoder

print(f"guided_diffusion: {create_model_and_diffusion.__name__}")
print(f"rcdm.dataset: {RepresentationDataset.__name__}")
print(f"rcdm.encoder: {load_encoder.__name__}")

# Check data file exists at the path expected by scripts/train.py
data_path = os.path.join(REPO_DIR, "data", "tiny-imagenet-200", "train_reps.pt")
assert os.path.exists(data_path), f"Data file not found: {data_path}"

print(f"\nAll imports OK")
print(f"Data file found: {os.path.getsize(data_path) / 1e9:.2f} GB")

guided_diffusion: create_model_and_diffusion
rcdm.dataset: RepresentationDataset
rcdm.encoder: load_encoder

All imports OK
Data file found: 0.83 GB


In [20]:
# (cell removed — verification is handled by Cell 15 above)


## 7. Start Training

Run your training script with GPU acceleration!

In [21]:
# Training with GPU
!python scripts/train.py \
    --reps_file data/tiny-imagenet-200/train_reps.pt \
    --save_dir /content/drive/MyDrive/rcdm_checkpoints \
    --image_size 64 \
    --batch_size 128 \
    --lr 1e-4 \
    --total_steps 50000 \
    --save_interval 2000 \
    --log_interval 100 \
    --num_workers 4 \
    --device cuda


[1/4] Building model...
  Model parameters: 112.0M

[2/4] Loading dataset...
Loading representations from data/tiny-imagenet-200/train_reps.pt...
  100000 image-representation pairs loaded
  100000 samples, 781 batches per epoch at batch_size=128

[3/4] Setting up optimiser...

[4/4] Starting training...

Traceback (most recent call last):
  File "/content/master_implementation/scripts/train.py", line 210, in <module>
    main()
  File "/content/master_implementation/scripts/train.py", line 136, in main
    x, h = next(data_iter)
           ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1548, in _next_data
    return self._process_data(data, worker_id)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader

## 8. Monitor Training (Optional - Run in separate cell)

In [22]:
# Check latest checkpoint
!ls -lh /content/drive/MyDrive/rcdm_checkpoints/

# Monitor GPU usage
!nvidia-smi

total 0
Mon Apr 27 08:34:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             85W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------------------

## 9. Generate Samples (After Training)

In [23]:
# Load your sampling script here once you create it
# !python scripts/sampling.py --checkpoint /content/drive/MyDrive/rcdm_checkpoints/model_100000.pt

---

## Tips

| Topic | Guidance |
|-------|----------|
| **GPU runtime limit** | Free Colab = ~12 h. Save checkpoints frequently (`--save_interval 2000`). |
| **Batch size** | T4: use 64–128. A100/H100: up to 256. |
| **Checkpoints** | Saved to Google Drive — survive session restarts. |
| **Code changes** | Edit in Cursor → `git push` → re-run Cell 3 (`git pull`) in Colab. No restart needed. |
| **Monitor GPU** | Run `!nvidia-smi` in a spare cell at any time. |

## Resume after disconnection

Re-run cells 1–6 (clone/pull, install, data), then:

```python
!python scripts/train.py \
    --reps_file data/tiny-imagenet-200/train_reps.pt \
    --save_dir /content/drive/MyDrive/rcdm_checkpoints \
    --image_size 64 \
    --batch_size 128 \
    --lr 1e-4 \
    --total_steps 50000 \
    --save_interval 2000 \
    --log_interval 100 \
    --num_workers 4 \
    --resume /content/drive/MyDrive/rcdm_checkpoints/rcdm_stepXXXXXXX.pt \
    --device cuda
```